# matmul-2d — worked example 3: Test Matmul Chain Associativity: (A@B)@C == A@(B@C)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-2d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Matrix multiplication is associative: `(A @ B) @ C = A @ (B @ C)`. The shape rule must hold at each step: if `A: (M, K)`, `B: (K, L)`, `C: (L, N)`, then both groupings produce shape `(M, N)`. Associativity is useful when choosing the order of contraction to minimize FLOPs — you can compute the cheaper pairing first.

## Worked solution

**Step 1 — choose compatible shapes.**
We use `A: (4, 3)`, `B: (3, 5)`, `C: (5, 2)`. The inner dimensions match at each step: 3 matches 3, and 5 matches 5. Final output shape: `(4, 2)`.

**Step 2 — compute left-grouping: (A @ B) @ C.**
First `A @ B` gives `(4, 5)`, then `@ C` gives `(4, 2)`. Intermediate: shape `(4, 5)`.

**Step 3 — compute right-grouping: A @ (B @ C).**
First `B @ C` gives `(3, 2)`, then `A @` gives `(4, 2)`. Intermediate: shape `(3, 2)`.

**Step 4 — verify both results are equal.**
Despite different intermediate shapes, the final `(4, 2)` outputs must agree numerically up to floating-point tolerance. This confirms associativity and demonstrates that you can choose evaluation order freely.

In [ ]:
import torch as t

t.manual_seed(13)
A = t.randn(4, 3)
B = t.randn(3, 5)
C = t.randn(5, 2)

# Left grouping: (A@B)@C
AB = A @ B          # (4,3)@(3,5) = (4,5)
ABC_left = AB @ C   # (4,5)@(5,2) = (4,2)

# Right grouping: A@(B@C)
BC = B @ C          # (3,5)@(5,2) = (3,2)
ABC_right = A @ BC  # (4,3)@(3,2) = (4,2)

print(f"A: {A.shape}, B: {B.shape}, C: {C.shape}")
print(f"(A@B)@C shape: {ABC_left.shape}")
print(f"A@(B@C) shape: {ABC_right.shape}")
print(f"Results equal: {t.allclose(ABC_left, ABC_right, atol=1e-5)}")

# FLOP counts: left grouping needs 4*3*5 + 4*5*2 = 60+40 = 100 mults
# Right grouping needs 3*5*2 + 4*3*2 = 30+24 = 54 mults -> cheaper!
print(f"\nLeft grouping:  ~100 multiplications")
print(f"Right grouping: ~54 multiplications (cheaper — do B@C first)")